In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0/val.csv
/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0/train.csv
/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0/test.csv
/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0/dataset.csv


In [2]:
!pip install transformers datasets torch scikit-learn -q

In [3]:
import pandas as pd

BASE = "/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0"
train_df = pd.read_csv(f"{BASE}/train.csv")
val_df   = pd.read_csv(f"{BASE}/val.csv")
test_df  = pd.read_csv(f"{BASE}/test.csv")
print(train_df.shape, val_df.shape, test_df.shape)

(2993, 9) (374, 9) (375, 9)


In [4]:
# Kiểm tra label_int trong val
print(val_df["label_int"].value_counts())
print(val_df["label_int"].isna().sum())
print(val_df["label_int"].dtype)

# Kiểm tra text
print(val_df["text"].isna().sum())

label_int
1    187
0    187
Name: count, dtype: int64
0
int64
0


In [5]:
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset

tokenizer = AutoTokenizer.from_pretrained("microsoft/mdeberta-v3-base")

class ViDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        encodings = tokenizer(
            list(df["text"]),
            truncation=True,
            padding=True,
            max_length=max_len
        )
        # Chỉ giữ input_ids và attention_mask, bỏ token_type_ids
        self.input_ids = encodings["input_ids"]
        self.attention_mask = encodings["attention_mask"]
        self.labels = list(df["label_int"])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx]),
            "attention_mask": torch.tensor(self.attention_mask[idx]),
            "labels": torch.tensor(self.labels[idx])
        }

train_ds = ViDataset(train_df, tokenizer)
val_ds   = ViDataset(val_df,   tokenizer)
test_ds  = ViDataset(test_df,  tokenizer)
print("Keys:", train_ds[0].keys())
print("Datasets ready:", len(train_ds), len(val_ds), len(test_ds))

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Datasets ready: 2993 374 375


In [6]:
# Kiểm tra tokenizer output
sample = tokenizer(
    list(val_df["text"][:2]),
    truncation=True,
    padding=True,
    max_length=512
)
print("Keys:", sample.keys())
print("input_ids shape:", len(sample["input_ids"]), len(sample["input_ids"][0]))

# Kiểm tra giá trị lạ
import numpy as np
ids = np.array(sample["input_ids"])
print("Min:", ids.min(), "Max:", ids.max())
print("NaN trong input_ids:", np.isnan(ids.astype(float)).any())

Keys: KeysView({'input_ids': [[532, 1410, 371, 797, 4543, 691, 1545, 267, 31021, 335, 318, 709, 263, 367, 1658, 267, 978, 1076, 262, 797, 260, 1181, 8882, 260, 264, 2161, 301, 1859, 260, 2625, 704, 3572, 332, 2295, 7791, 371, 891, 5037, 708, 2056, 262, 356, 1991, 260, 28335, 263, 1313, 421, 260, 29540, 274, 260, 794, 719, 2343, 260, 319, 56303, 260, 43587, 331, 2833, 371, 4599, 24193, 267, 261, 1517, 1303, 260, 273, 9268, 262, 395, 2044, 356, 5310, 759, 8882, 2077, 395, 1850, 535, 1293, 319, 3124, 260, 287, 1410, 371, 301, 3080, 273, 302, 5972, 1102, 260, 271, 1410, 371, 260, 3666, 262, 260, 271, 1410, 371, 719, 4032, 262, 838, 1406, 260, 273, 1993, 318, 3922, 260, 287, 1410, 371, 31021, 335, 5987, 3457, 395, 3256, 260, 273, 1993, 327, 5080, 356, 1076, 318, 2833, 273, 318, 3173, 267, 261, 10831, 318, 1843, 371, 331, 2592, 260, 110766, 3124, 260, 287, 1410, 371, 797, 260, 271, 1243, 395, 2290, 271, 318, 1545, 260, 4889, 262, 260, 273, 9268, 332, 2242, 260, 2181, 678, 1991, 1673, 301, 30

In [7]:
print(train_ds[0].keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [8]:
# Kiểm tra label trong dataset
print(train_ds[0]["labels"])
print(train_ds[1]["labels"])

# Kiểm tra thử 1 batch
from torch.utils.data import DataLoader
loader = DataLoader(train_ds, batch_size=2)
batch = next(iter(loader))
print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("labels:", batch["labels"])
print("labels dtype:", batch["labels"].dtype)

tensor(0)
tensor(0)
input_ids: torch.Size([2, 512])
attention_mask: torch.Size([2, 512])
labels: tensor([0, 0])
labels dtype: torch.int64


In [9]:
from transformers import (AutoModelForSequenceClassification,
                           TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/mdeberta-v3-base",
    num_labels=2,
    dtype=torch.float32  # ← đổi torch_dtype thành dtype
)
model = model.float()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="binary")
    }

args = TrainingArguments(
    output_dir="./mdeberta-ai-detection-v2",
    num_train_epochs=5,
    per_device_train_batch_size=4,   # ← giảm từ 8 xuống 4
    per_device_eval_batch_size=8,    # ← giảm từ 16 xuống 8
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=False,
    bf16=False,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
classifier.weight                  

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.190737,0.372160,0.959893,0.958449
2,0.135799,0.262109,0.981283,0.980926
3,0.130325,0.408371,0.970588,0.971279
4,0.001799,0.172360,0.983957,0.984043
5,0.016449,0.188296,0.986631,0.986450


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1875, training_loss=0.20353240957359472, metrics={'train_runtime': 1721.4946, 'train_samples_per_second': 8.693, 'train_steps_per_second': 1.089, 'total_flos': 3937527557191680.0, 'train_loss': 0.20353240957359472, 'epoch': 5.0})

In [10]:
from scipy.special import softmax
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import numpy as np

# Predict trên test set
pred_output = trainer.predict(test_ds)
preds  = np.argmax(pred_output.predictions, axis=-1)
probs  = softmax(pred_output.predictions, axis=-1)[:, 1]
labels = pred_output.label_ids

print("=== Test Metrics ===")
print(f"Accuracy : {accuracy_score(labels, preds):.4f}")
print(f"F1 (macro): {f1_score(labels, preds, average='macro'):.4f}")
print(f"AUROC    : {roc_auc_score(labels, probs):.4f}")

print("\n=== Classification Report ===")
print(classification_report(labels, preds, target_names=["human", "ai"]))

print("=== Confusion Matrix ===")
print(confusion_matrix(labels, preds))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


=== Test Metrics ===
Accuracy : 0.9840
F1 (macro): 0.9840
AUROC    : 0.9993

=== Classification Report ===
              precision    recall  f1-score   support

       human       0.99      0.98      0.98       188
          ai       0.98      0.99      0.98       187

    accuracy                           0.98       375
   macro avg       0.98      0.98      0.98       375
weighted avg       0.98      0.98      0.98       375

=== Confusion Matrix ===
[[184   4]
 [  2 185]]


In [11]:
import shutil
import os

# Xóa các checkpoint trung gian (chỉ giữ model cuối)
checkpoint_dir = "./mdeberta-ai-detection-v2"
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
    print("Đã xóa checkpoints!")

# Kiểm tra dung lượng còn lại
import subprocess
result = subprocess.run(['du', '-sh', '/kaggle/working'], capture_output=True, text=True)
print("Dung lượng hiện tại:", result.stdout)

Đã xóa checkpoints!
Dung lượng hiện tại: 52K	/kaggle/working



In [12]:
trainer.save_model("/kaggle/working/mdeberta-finetuned")
tokenizer.save_pretrained("/kaggle/working/mdeberta-finetuned")
print("Saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved!


In [13]:
import os
for f in os.listdir("/kaggle/working/mdeberta-finetuned"):
    size = os.path.getsize(f"/kaggle/working/mdeberta-finetuned/{f}")
    print(f"{f}: {size/1024/1024:.1f} MB")

model.safetensors: 1063.6 MB
config.json: 0.0 MB
tokenizer_config.json: 0.0 MB
tokenizer.json: 15.3 MB
training_args.bin: 0.0 MB
